# Phase 4 : transformation et feature engineering

Projet final Machine Learning, blocs 6 et 8.

## Objectif de ce notebook

Combiner les trois sources et construire les variables dérivées qui feront le modèle. Le jeu
nettoyé de la phase 3 est propre mais brut : il décrit un état de jeu, pas un contexte.

## La règle qui domine toute cette phase

Quatre features regardent le passé : la forme d'une équipe, l'expérience d'un roster, le taux
de victoire d'un champion et la médiane d'or du patch. Elles doivent être calculées
**uniquement sur les parties antérieures**, triées par date.

Un `groupby().mean()` global donnerait à chaque équipe ses propres résultats futurs. Le modèle
paraîtrait excellent jusqu'au jour où on l'utilise vraiment, et c'est exactement la fuite qui
plafonne la phase de modélisation à 8 sur 25.

Deux mécanismes seulement sont employés, tous deux dans `src/features.py` :

- `shift(1)` après tri par date, pour qu'une fenêtre glissante s'arrête à la partie
  précédente ;
- `cumcount()` et un `cumsum()` décalé, qui ne regardent le passé que par construction.

La section 5.5 vérifie ce principe par un audit plutôt que de le supposer.

## Livrables

`data/processed/lol_at15.parquet` et `docs/data_dictionary.md`.

## 0. Chargement

On repart de `data/interim/`, produit par la phase 3, et jamais de `data/raw/`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import analyse, config, extraction, features

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

equipes = pd.read_parquet(config.DATA_INTERIM / "equipes_interim.parquet")
joueurs = pd.read_parquet(config.DATA_INTERIM / "joueurs_interim.parquet")
champions = extraction.load_champions()

LIGNES_ATTENDUES = len(equipes)

print()
print(f"Lignes équipe : {len(equipes):,} x {equipes.shape[1]} colonnes")
print(f"Lignes joueur : {len(joueurs):,} x {joueurs.shape[1]} colonnes")
print(f"Champions     : {len(champions):,}")
print(f"Période       : {equipes['date'].min():%Y-%m-%d} à {equipes['date'].max():%Y-%m-%d}")

Loaded 173 champions, tags: ['Assassin', 'Fighter', 'Mage', 'Marksman', 'Support', 'Tank']
5 champions have a French name different from the English join key

Lignes équipe : 92,616 x 66 colonnes
Lignes joueur : 463,080 x 8 colonnes
Champions     : 173
Période       : 2022-01-10 à 2026-09-06


## 1. Jointures

Le guide demande de documenter chaque jointure. Ce projet en compte trois, dont deux ici, la
troisième ayant été faite en phase 3.

| # | Table gauche | Table droite | Clé | Type | Cardinalité |
|---|---|---|---|---|---|
| 1 | Lignes équipe | Référentiel des ligues | `league` | left | plusieurs vers un |
| 2 | Lignes joueur | Data Dragon | `champion` | left | plusieurs vers un |
| 3 | Lignes équipe | Composition agrégée | `gameid` + `side` | left | un vers un |

### Le choix de la clé, qui n'est pas anodin

La clé naturelle serait `gameid` + `teamid`. Elle est mauvaise ici : la phase 2 a montré que
`teamid` manque sur 1 800 lignes, et la phase 3 les a remplies par la constante `Inconnu`.
Regrouper là-dessus fusionnerait les deux équipes d'une même partie sous une clé identique.

`gameid` + `side` n'a pas ce défaut : `side` vaut toujours `Blue` ou `Red`, sans exception, et
distingue donc toujours les deux camps.

In [2]:
# Contrôle des clés avant de joindre, comme le demande la checklist du guide.
print("Unicité de gameid + side sur les lignes équipe :",
      not equipes.duplicated(subset=["gameid", "side"]).any())
print("Lignes joueur par couple gameid + side :",
      joueurs.groupby(["gameid", "side"]).size().value_counts().to_dict())
print()
print("Valeurs manquantes sur les clés :")
print(f"  equipes[gameid, side] : {int(equipes[['gameid', 'side']].isna().sum().sum())}")
print(f"  joueurs[gameid, side] : {int(joueurs[['gameid', 'side']].isna().sum().sum())}")
print(f"  joueurs[champion]     : {int(joueurs['champion'].isna().sum())}")
print()
print("Rappel du problème évité :")
print(f"  lignes équipe à teamid 'Inconnu' : {int((equipes['teamid'] == 'Inconnu').sum()):,}")

Unicité de gameid + side sur les lignes équipe : True
Lignes joueur par couple gameid + side : {5: 92616}

Valeurs manquantes sur les clés :
  equipes[gameid, side] : 0


  joueurs[gameid, side] : 0
  joueurs[champion]     : 0

Rappel du problème évité :
  lignes équipe à teamid 'Inconnu' : 1,734


## 2. Features de composition

Les cinq champions d'une équipe sont reconstruits depuis les **lignes joueur**, pas depuis les
colonnes `pick1` à `pick5`.

La raison est mesurée : `pick1` manque sur 9 % des lignes équipe, alors que `champion` ne
manque sur aucune ligne joueur. Les lignes joueur sont en plus la seule source qui associe un
champion à un poste.

Un champion compte pour un tag qu'il soit principal ou secondaire. Nunu et Willump est Tank et
Mage : ne retenir que le premier tag perdrait la moitié de l'information.

In [3]:
composition = features.composition_par_equipe(joueurs, champions)

print("Composition construite :", composition.shape)
print()
print("Champions reconnus par équipe, 5 attendu :")
print(composition["champions_reconnus"].value_counts().to_string())
print()
colonnes_compo = [c for c in composition.columns if c.startswith("compo_nb_")]
print("Distribution des compositions :")
print(composition[colonnes_compo + ["profil_degats"]].describe().T[
    ["mean", "min", "50%", "max"]].round(2).to_string())

Composition construite : (92616, 10)

Champions reconnus par équipe, 5 attendu :
champions_reconnus
5    92616

Distribution des compositions :
                   mean  min   50%  max
compo_nb_tank      1.62  0.0  2.00  4.0
compo_nb_mage      1.85  0.0  2.00  5.0
compo_nb_marksman  1.29  0.0  1.00  4.0
compo_nb_fighter   1.70  0.0  2.00  5.0
compo_nb_assassin  0.92  0.0  1.00  5.0
compo_nb_support   1.38  0.0  1.00  4.0
profil_degats      0.75  0.0  0.75  1.0


Les cinq champions sont reconnus sur la totalité des équipes-parties. La correction de phase 1,
qui joint sur le nom anglais et non sur le nom français, porte ici ses fruits : sans elle, les
compositions contenant Nunu et Willump, K'Sante, Maître Yi, Séraphine ou Zoé auraient été
comptées avec un champion de moins.

`profil_degats` est exprimé comme une **part** et non comme un rapport AD sur AP. Un rapport
diviserait par zéro sur les compositions sans le moindre mage, ce qui arrive.

In [4]:
roster = features.roster_par_equipe(joueurs)
print("Rosters construits :", roster.shape)
print("Compositions de joueurs distinctes :", roster["roster_key"].nunique())
print()

avant = len(equipes)
equipes = equipes.merge(composition, on=["gameid", "side"], how="left", validate="one_to_one")
equipes = equipes.merge(roster, on=["gameid", "side"], how="left", validate="one_to_one")

print(f"Lignes avant jointures : {avant:,} | après : {len(equipes):,}")
print(f"Manquants sur la composition : {int(equipes[colonnes_compo].isna().sum().sum())}")
print(f"Manquants sur roster_key     : {int(equipes['roster_key'].isna().sum())}")

Rosters construits : (92616, 3)
Compositions de joueurs distinctes : 7375



Lignes avant jointures : 92,616 | après : 92,616
Manquants sur la composition : 0
Manquants sur roster_key     : 0


## 3. Features calculées

Opérations directes entre colonnes déjà présentes, sans aucune statistique apprise sur les
données. Elles ne posent donc aucun risque de fuite.

In [5]:
equipes["diff_kills_at15"] = equipes["killsat15"] - equipes["opp_killsat15"]

# Trois objectifs et non quatre : la phase 3 a écarté firsttower, qui est un
# drapeau de fin de partie et non un état à la minute 15.
equipes["objectifs_precoces"] = equipes[config.EARLY_OBJECTIVES].sum(axis=1, min_count=1)

equipes["is_playoffs"] = equipes["playoffs"].fillna(0).astype(int)

print("Objectifs sommés :", config.EARLY_OBJECTIVES)
print()
print("Distribution de objectifs_precoces, 0 à 3 attendu :")
print(equipes["objectifs_precoces"].value_counts(dropna=False).sort_index().to_string())
print()
print("diff_kills_at15 :")
print(equipes["diff_kills_at15"].describe().round(1).to_string())

Objectifs sommés : ['firstblood', 'firstdragon', 'firstherald']

Distribution de objectifs_precoces, 0 à 3 attendu :
objectifs_precoces
0    14169
1    32279
2    32158
3    14010

diff_kills_at15 :
count    92616.0
mean         0.0
std          3.8
min        -24.0
25%         -2.0
50%          0.0
75%          2.0
max         24.0


## 4. Découpage du patch et rang dans la saison

`patch_major` ne peut pas être une feature : ses valeurs de 2026 n'apparaissent jamais à
l'entraînement, et un encodage les réduirait à un bloc de zéros. Le **rang** du patch dans sa
saison, lui, généralise : chaque saison a un premier patch, un deuxième, et ainsi de suite.

In [6]:
equipes[["patch_major", "patch_minor"]] = features.decouper_patch(equipes["patch"])
equipes["patch_seq"] = features.rang_patch_dans_saison(equipes)

print("Patchs par saison, et rang maximal :")
print(equipes.groupby("saison").agg(
    patchs=("patch", "nunique"), rang_max=("patch_seq", "max")).to_string())
print()
print("Manquants sur patch_seq :", int(equipes["patch_seq"].isna().sum()))
print()
print("Contrôle du piège de tri, patchs de 2022 dans l'ordre :")
apercu = (equipes[equipes["saison"] == 2022]
          [["patch", "patch_major", "patch_minor", "patch_seq"]]
          .drop_duplicates().sort_values("patch_seq").head(8))
print(apercu.to_string(index=False))

Patchs par saison, et rang maximal :
        patchs  rang_max
saison                  
2022        22        22
2023        21        21
2024        23        23
2025        24        24
2026        17        17

Manquants sur patch_seq : 0

Contrôle du piège de tri, patchs de 2022 dans l'ordre :
patch  patch_major  patch_minor  patch_seq
12.01           12            1          1
12.02           12            2          2
12.03           12            3          3
12.04           12            4          4
12.05           12            5          5
12.06           12            6          6
12.07           12            7          7
12.08           12            8          8


## 5. Features historiques, le point sensible

Les quatre features qui suivent regardent le passé. Ce sont les seules du projet qui peuvent
introduire une fuite, et elles la produiraient en silence.

### 5.1 Clé d'équipe

`teamid` manque sur une partie des lignes et vaut `Inconnu` depuis la phase 3. Utilisée telle
quelle, elle regrouperait sous une même clé toutes les équipes non identifiées, dont la forme
n'aurait aucun sens.

On construit donc `team_key` : `teamid` quand il existe, `teamname` sinon.

In [7]:
equipes["team_key"] = equipes["teamid"].where(
    equipes["teamid"].ne("Inconnu"), equipes["teamname"]
)

print(f"teamid distincts   : {equipes['teamid'].nunique():,}")
print(f"team_key distincts : {equipes['team_key'].nunique():,}")
print(f"Lignes retombées sur teamname : {int((equipes['teamid'] == 'Inconnu').sum()):,}")

teamid distincts   : 1,140
team_key distincts : 1,386
Lignes retombées sur teamname : 1,734


### 5.2 Forme récente et expérience du roster

`forme_equipe_10_derniers` est le taux de victoire sur les dix parties précédentes. Le
`shift(1)` fait toute la différence : sans lui, le résultat de la partie courante entrerait
dans son propre prédicteur.

`experience_roster` compte les parties déjà jouées ensemble par ces cinq joueurs. Un
`cumcount` ne compte que les lignes qui précèdent, il est donc antérieur par nature.

In [8]:
equipes["forme_equipe_10_derniers"] = features.forme_equipe(equipes, fenetre=10)
equipes["experience_roster"] = features.experience_roster(equipes)

print("forme_equipe_10_derniers")
print(f"  NaN     : {int(equipes['forme_equipe_10_derniers'].isna().sum()):,} "
      f"(première partie de chaque équipe)")
print(f"  moyenne : {equipes['forme_equipe_10_derniers'].mean():.4f} (0.5 attendu)")
print()
print("experience_roster")
print(f"  médiane : {equipes['experience_roster'].median():.0f} | "
      f"max : {equipes['experience_roster'].max()}")
print(f"  rosters inédits : {int((equipes['experience_roster'] == 0).sum()):,}")

forme_equipe_10_derniers
  NaN     : 1,386 (première partie de chaque équipe)
  moyenne : 0.5104 (0.5 attendu)

experience_roster
  médiane : 10 | max : 425
  rosters inédits : 7,375


### 5.3 Taux de victoire des champions sur le patch

Calculé au niveau des lignes joueur, où une ligne est un champion dans une partie, puis moyenné
sur les cinq champions de l'équipe.

Le numérateur est une somme cumulée dont on retranche la ligne courante, le dénominateur un
`cumcount`. Les deux sont antérieurs. La première apparition d'un champion sur un patch n'a pas
d'historique et vaut `NaN`, ce qui est la réponse honnête : la remplir par 0,5 injecterait une
hypothèse.

In [9]:
# On rattache à chaque ligne joueur le résultat, la date et le patch de son équipe.
cle_equipe = equipes[["gameid", "side", "result", "date", "patch"]]
joueurs_res = joueurs[["gameid", "side", "champion", "playername"]].merge(
    cle_equipe, on=["gameid", "side"], how="left", validate="many_to_one"
)
print(f"Lignes joueur enrichies : {len(joueurs_res):,} | "
      f"résultat manquant : {int(joueurs_res['result'].isna().sum())}")

joueurs_res["wr_champion"] = features.winrate_champion_patch(joueurs_res)

print(f"  NaN     : {int(joueurs_res['wr_champion'].isna().sum()):,} "
      f"({100 * joueurs_res['wr_champion'].isna().mean():.1f} %)")
print(f"  moyenne : {joueurs_res['wr_champion'].mean():.4f} (0.5 attendu)")
print()

agrege = (joueurs_res.groupby(["gameid", "side"])["wr_champion"]
          .mean().rename("winrate_champion_patch").reset_index())
equipes = equipes.merge(agrege, on=["gameid", "side"], how="left", validate="one_to_one")

print(f"Après agrégation par équipe, NaN : "
      f"{int(equipes['winrate_champion_patch'].isna().sum()):,}")

Lignes joueur enrichies : 463,080 | résultat manquant : 0


  NaN     : 13,773 (3.0 %)
  moyenne : 0.5032 (0.5 attendu)



Après agrégation par équipe, NaN : 425


### 5.4 Écart d'or normalisé

La phase 2 a mesuré une dérive de méta de 7 à 10 % sur l'or entre l'entraînement et le test. Un
avantage de 2 000 or ne vaut donc pas la même chose en 2022 et en 2026, et il faut le rapporter
au niveau économique de son patch.

**C'est ici que se cachait le piège le plus discret de la phase.** L'implémentation naturelle,
`groupby("patch")["goldat15"].transform("median")`, donne à chaque partie la médiane de tout
son patch, parties postérieures comprises. C'est une fuite, moins visible que les autres parce
qu'elle passe par une statistique agrégée et non par une colonne suspecte.

La médiane est donc calculée en fenêtre expansive sur les parties antérieures du même patch.

In [10]:
equipes["mediane_or_patch"] = features.mediane_or_patch_expansive(equipes, min_parties=20)
equipes["ecart_or_normalise"] = equipes["golddiffat15"] / equipes["mediane_or_patch"]

print(f"mediane_or_patch NaN : {int(equipes['mediane_or_patch'].isna().sum()):,} "
      f"(amorçage des 20 premières parties de chaque patch)")
print()
print("ecart_or_normalise :")
print(equipes["ecart_or_normalise"].describe().round(3).to_string())
print()
print("Dispersion, entraînement contre test :")
for nom, sous in [("train", equipes[equipes["date"] < config.SPLIT_DATE]),
                  ("test ", equipes[equipes["date"] >= config.SPLIT_DATE])]:
    print(f"  {nom} : golddiffat15 std = {sous['golddiffat15'].std():>8.1f} | "
          f"ecart_or_normalise std = {sous['ecart_or_normalise'].std():.4f}")

mediane_or_patch NaN : 2,134 (amorçage des 20 premières parties de chaque patch)



ecart_or_normalise :
count    90482.000
mean         0.000
std          0.120
min         -0.670
25%         -0.073
50%          0.000
75%          0.073
max          0.670

Dispersion, entraînement contre test :
  train : golddiffat15 std =   2985.2 | ecart_or_normalise std = 0.1205
  test  : golddiffat15 std =   3120.5 | ecart_or_normalise std = 0.1182


### 5.5 Audit de fuite temporelle

Une feature antérieure n'a rien à dire sur la première partie d'une équipe ou d'un champion. Si
elle parle quand même, c'est qu'elle a lu le présent.

Ce test ne prouve pas l'absence de fuite, mais il attrape la classe d'erreur la plus courante,
l'oubli du `shift`, pour un coût négligeable.

In [11]:
audit_equipe = features.auditer_fuite_temporelle(
    equipes, ["forme_equipe_10_derniers"], cle="team_key")
audit_champion = features.auditer_fuite_temporelle(
    joueurs_res, ["wr_champion"], cle="champion")

audit = pd.concat([audit_equipe, audit_champion], ignore_index=True)
print(audit.to_string(index=False))
print()

# experience_roster vaut 0, et non NaN, sur une première partie : c'est la bonne
# valeur, un roster inédit a bien zéro partie commune derrière lui.
ordonne = equipes.sort_values(["date", "gameid"])
premiere = ordonne.groupby("roster_key").cumcount() == 0
print("experience_roster sur une première partie, valeurs distinctes :",
      sorted(ordonne.loc[premiere, "experience_roster"].unique()), "(0 attendu)")

                 feature cle_de_groupe  premieres_parties  renseignees_a_tort verdict
forme_equipe_10_derniers      team_key               1386                   0      ok
             wr_champion      champion                173                   0      ok

experience_roster sur une première partie, valeurs distinctes : [0] (0 attendu)


In [12]:
# Contrôle complémentaire : une feature antérieure ne doit pas corréler avec la
# cible plus fort que le meilleur signal disponible à la minute 15.
features_historiques = ["forme_equipe_10_derniers", "experience_roster",
                        "winrate_champion_patch", "ecart_or_normalise"]
correlations = equipes[features_historiques + ["golddiffat15", "result"]].corr()["result"].abs()

print("Corrélation absolue avec result :")
print(correlations.drop("result").sort_values(ascending=False).round(3).to_string())
print()
print(f"Plafond légitime, golddiffat15 : {correlations['golddiffat15']:.3f}")

Corrélation absolue avec result :
ecart_or_normalise          0.536
golddiffat15                0.535
forme_equipe_10_derniers    0.181
experience_roster           0.058
winrate_champion_patch      0.031

Plafond légitime, golddiffat15 : 0.535


Les features historiques corrèlent faiblement avec la cible, bien en dessous de l'écart d'or.
C'est attendu et rassurant : la forme récente d'une équipe explique une part réelle mais
modeste du résultat. Une corrélation forte ici aurait signalé une fuite.

## 6. Features temporelles et sélection finale

Le guide demande des features temporelles quand des dates sont disponibles. `saison` et
`patch_seq` en sont déjà. On ajoute le mois, utile à l'analyse de phase 5, mais qui ne sera pas
donné au modèle : un mois de calendrier n'a pas de sens prédictif ici et n'apporterait que du
bruit.

In [13]:
equipes["mois"] = equipes["date"].dt.month

COLONNES_FINALES = (
    ["gameid", "teamid", "team_key", "teamname", "date", "saison", "league", "patch"]
    + ["result"]
    + config.NUMERIC_FEATURES
    + config.CATEGORICAL_FEATURES
    + config.BOOLEAN_FEATURES
    + ["is_playoffs", "mois", "year", "patch_major", "patch_minor",
       "compo_nb_assassin", "compo_nb_support", "turretplates",
       "flag_ecart_or_extreme", "confiance", "franchisee", "mediane_or_patch",
       "roster_key", "goldat15", "opp_goldat15", "xpat15", "csat15"]
)
COLONNES_FINALES = list(dict.fromkeys(COLONNES_FINALES))

manquantes = [c for c in COLONNES_FINALES if c not in equipes.columns]
print("Colonnes attendues et absentes :", manquantes)

final = equipes[COLONNES_FINALES].copy()
print(f"Dataset final : {final.shape[0]:,} lignes x {final.shape[1]} colonnes")
print()
print("Features données au modèle :")
print(f"  numériques    ({len(config.NUMERIC_FEATURES)}) : {config.NUMERIC_FEATURES}")
print(f"  catégorielles ({len(config.CATEGORICAL_FEATURES)}) : {config.CATEGORICAL_FEATURES}")
print(f"  booléennes    ({len(config.BOOLEAN_FEATURES)}) : {config.BOOLEAN_FEATURES}")

Colonnes attendues et absentes : []
Dataset final : 92,616 lignes x 49 colonnes

Features données au modèle :
  numériques    (16) : ['golddiffat15', 'xpdiffat15', 'csdiffat15', 'diff_kills_at15', 'deathsat15', 'objectifs_precoces', 'compo_nb_tank', 'compo_nb_mage', 'compo_nb_marksman', 'compo_nb_fighter', 'profil_degats', 'forme_equipe_10_derniers', 'experience_roster', 'winrate_champion_patch', 'ecart_or_normalise', 'patch_seq']
  catégorielles (3) : ['side', 'region', 'tier_ligue']
  booléennes    (4) : ['playoffs', 'firstblood', 'firstdragon', 'firstherald']


## 7. Validation

In [14]:
controles = [
    ("Aucune ligne perdue ni dupliquée par les jointures",
     len(final) == LIGNES_ATTENDUES),
    ("Deux lignes par partie, sans exception",
     bool(final.groupby("gameid").size().eq(2).all())),
    ("Cible toujours parfaitement équilibrée", float(final["result"].mean()) == 0.5),
    ("Toutes les features du modèle sont présentes",
     all(c in final.columns for c in
         config.NUMERIC_FEATURES + config.CATEGORICAL_FEATURES + config.BOOLEAN_FEATURES)),
    ("objectifs_precoces borné entre 0 et 3",
     bool(final["objectifs_precoces"].dropna().between(0, 3).all())),
    ("profil_degats borné entre 0 et 1",
     bool(final["profil_degats"].dropna().between(0, 1).all())),
    ("forme_equipe bornée entre 0 et 1",
     bool(final["forme_equipe_10_derniers"].dropna().between(0, 1).all())),
    ("winrate_champion_patch borné entre 0 et 1",
     bool(final["winrate_champion_patch"].dropna().between(0, 1).all())),
    ("patch_seq entièrement renseigné", int(final["patch_seq"].isna().sum()) == 0),
    ("Aucune feature historique renseignée sur une première partie",
     bool((audit["renseignees_a_tort"] == 0).all())),
    ("Aucune colonne de fuite dans le dataset final",
     not any(c in final.columns for c in config.LEAKY_COLUMNS)),
    ("Le split chronologique reste faisable",
     int((final["date"] < config.SPLIT_DATE).sum()) > 0
     and int((final["date"] >= config.SPLIT_DATE).sum()) > 0),
]

bilan = pd.DataFrame(controles, columns=["controle", "statut"])
bilan["statut"] = bilan["statut"].map({True: "ok", False: "a corriger"})
print(bilan.to_string(index=False))
print()
print("Contrôles en échec :", int((bilan["statut"] == "a corriger").sum()))

                                                    controle statut
          Aucune ligne perdue ni dupliquée par les jointures     ok
                      Deux lignes par partie, sans exception     ok
                      Cible toujours parfaitement équilibrée     ok
                Toutes les features du modèle sont présentes     ok
                       objectifs_precoces borné entre 0 et 3     ok
                            profil_degats borné entre 0 et 1     ok
                            forme_equipe bornée entre 0 et 1     ok
                   winrate_champion_patch borné entre 0 et 1     ok
                             patch_seq entièrement renseigné     ok
Aucune feature historique renseignée sur une première partie     ok
               Aucune colonne de fuite dans le dataset final     ok
                       Le split chronologique reste faisable     ok

Contrôles en échec : 0


In [15]:
features_modele = (config.NUMERIC_FEATURES + config.CATEGORICAL_FEATURES
                   + config.BOOLEAN_FEATURES)
manquants = pd.DataFrame({
    "n_manquant": final[features_modele].isna().sum(),
    "pct": (100 * final[features_modele].isna().mean()).round(2),
}).sort_values("pct", ascending=False)

print("Valeurs manquantes sur les features du modèle :")
print(manquants[manquants["n_manquant"] > 0].to_string())

Valeurs manquantes sur les features du modèle :


                          n_manquant   pct
ecart_or_normalise              2134  2.30
forme_equipe_10_derniers        1386  1.50
winrate_champion_patch           425  0.46


Les valeurs manquantes qui subsistent sont toutes des **amorçages** : une équipe qui joue sa
première partie n'a pas de forme, un champion vu pour la première fois sur un patch n'a pas de
taux de victoire, un patch a besoin de vingt parties avant que sa médiane d'or veuille dire
quelque chose.

Elles ne sont pas imputées ici, et c'est délibéré. Le guide l'interdit avant le split, et un
`SimpleImputer` placé dans le `Pipeline` de la phase 7, ajusté sur le seul jeu d'entraînement,
fera le travail correctement. `HistGradientBoostingClassifier` sait de toute façon les traiter
nativement.

## 8. Tableau récapitulatif des features

Vue d'ensemble des variables réellement données au modèle en phase 7 : leur famille, leur
origine, la façon dont elles sont construites et leur pouvoir discriminant pris isolément.

Trois points de lecture.

**L'AUC univariée est calculée sur le seul jeu d'entraînement**, saisons 2022 à 2025. La saison
2026 n'est pas ouverte ici. Choisir ou écarter une feature en regardant le jeu de test
reviendrait à l'utiliser deux fois, et la mesure de performance finale ne vaudrait plus rien.

**Une AUC univariée de 0,50 ne disqualifie pas une variable.** Une ligne du dataset est une
équipe dans une partie, et les deux équipes d'une même partie partagent leur ligue, leur patch
et leur phase de compétition. Ces variables prennent donc la même valeur sur une ligne gagnante
et sur une ligne perdante : elles ne peuvent pas séparer les classes seules, quelle que soit
leur importance réelle. La colonne `symetrique` identifie ces cas, mesurés et non supposés.
Leur apport ne peut apparaître qu'en interaction, ce que la phase 7 tranchera.

**Une AUC univariée élevée ne prouve pas un apport.** `ecart_or_normalise` et `golddiffat15`
affichent presque le même score parce que la première dérive de la seconde. Une mesure
univariée ignore par construction toute redondance entre variables.


In [16]:
# Le tableau est construit à partir de config, jamais d'une liste recopiée : si une feature
# est ajoutée ou retirée de src/config.py, ce récapitulatif suit automatiquement.
FAMILLES = {
    "Etat de jeu a 15 min": ["golddiffat15", "xpdiffat15", "csdiffat15",
                             "diff_kills_at15", "deathsat15"],
    "Objectifs precoces": ["objectifs_precoces", "firstblood", "firstdragon", "firstherald"],
    "Composition de draft": ["compo_nb_tank", "compo_nb_mage", "compo_nb_marksman",
                             "compo_nb_fighter", "profil_degats"],
    "Historique, passe seul": ["forme_equipe_10_derniers", "experience_roster",
                               "winrate_champion_patch", "ecart_or_normalise"],
    "Contexte": ["patch_seq", "side", "region", "tier_ligue", "playoffs"],
}

SOURCES = {
    "Oracle's Elixir": ["golddiffat15", "xpdiffat15", "csdiffat15", "deathsat15",
                        "firstblood", "firstdragon", "firstherald", "side", "playoffs"],
    "Data Dragon": ["compo_nb_tank", "compo_nb_mage", "compo_nb_marksman",
                    "compo_nb_fighter", "profil_degats"],
    "Referentiel XLSX": ["region", "tier_ligue"],
}

CONSTRUCTIONS = {
    "diff_kills_at15": "killsat15 moins opp_killsat15",
    "objectifs_precoces": "somme de 3 drapeaux, 0 a 3",
    "compo_nb_tank": "comptage des tags sur les 5 picks",
    "compo_nb_mage": "comptage des tags sur les 5 picks",
    "compo_nb_marksman": "comptage des tags sur les 5 picks",
    "compo_nb_fighter": "comptage des tags sur les 5 picks",
    "profil_degats": "part AD parmi AD plus AP",
    "forme_equipe_10_derniers": "fenetre glissante 10, shift(1)",
    "experience_roster": "cumcount sur la cle de roster",
    "winrate_champion_patch": "cumsum decalee par champion et patch",
    "ecart_or_normalise": "or divise par mediane expansive du patch",
    "patch_seq": "rang du patch dans la saison",
    "region": "jointure sur le code ligue",
    "tier_ligue": "jointure sur le code ligue",
}

TYPES = ({f: "numerique" for f in config.NUMERIC_FEATURES}
         | {f: "categorielle" for f in config.CATEGORICAL_FEATURES}
         | {f: "booleenne" for f in config.BOOLEAN_FEATURES})

features_modele = list(TYPES)

# Pouvoir discriminant mesure sur l'entrainement seul. `side` est binaire mais textuelle :
# on la lit comme l'indicatrice du cote bleu, sans quoi l'AUC n'est pas definie.
train_recap = final[final["date"] < config.SPLIT_DATE].copy()
train_recap["side"] = (train_recap["side"] == "Blue").astype(int)
discriminant = analyse.pouvoir_discriminant(train_recap, features_modele).set_index("feature")

# Symetrie : une feature vaut-elle la meme chose sur les deux lignes d'une meme partie ?
part_symetrique = train_recap.groupby("gameid")[features_modele].nunique().eq(1).mean()

lignes = []
for famille, colonnes in FAMILLES.items():
    for col in colonnes:
        source = next((s for s, cols in SOURCES.items() if col in cols), "Derivee")
        lignes.append({
            "famille": famille,
            "feature": col,
            "type": TYPES[col],
            "source": source,
            "construction": CONSTRUCTIONS.get(col, "colonne brute"),
            "manquant_pct": round(100 * final[col].isna().mean(), 2),
            "auc_univariee": round(discriminant["auc_univarie"].get(col, float("nan")), 3),
            "sens": discriminant["sens"].get(col, "-"),
            "symetrique": "oui" if part_symetrique[col] > 0.99 else "non",
        })

recap = pd.DataFrame(lignes)

# `region` est textuelle et non ordonnee : une AUC n'y est pas definie. On l'affiche comme
# telle plutot que de laisser un NaN, qui se lirait comme une donnee manquante.
recap["auc_univariee"] = recap["auc_univariee"].map(
    lambda v: "n.a." if pd.isna(v) else f"{v:.3f}"
)

# Garde-fou : le tableau doit decrire exactement les features du modele, ni plus ni moins.
assert set(recap["feature"]) == set(features_modele), (
    "Le recapitulatif a diverge de config : "
    f"{set(features_modele) ^ set(recap['feature'])}"
)

print(f"{len(recap)} features donnees au modele : {recap['type'].value_counts().to_dict()}")
effectif_train = f"{len(train_recap):,}".replace(",", " ")  # separateur francais
print(f"AUC univariee mesuree sur {effectif_train} lignes d'entrainement, "
      f"saisons {train_recap['saison'].min()} a {train_recap['saison'].max()}")
print()
print(recap.to_string(index=False))


23 features donnees au modele : {'numerique': 16, 'booleenne': 4, 'categorielle': 3}
AUC univariee mesuree sur 76 070 lignes d'entrainement, saisons 2022 a 2025

               famille                  feature         type           source                             construction  manquant_pct auc_univariee    sens symetrique
  Etat de jeu a 15 min             golddiffat15    numerique  Oracle's Elixir                            colonne brute          0.00         0.820 positif        non
  Etat de jeu a 15 min               xpdiffat15    numerique  Oracle's Elixir                            colonne brute          0.00         0.792 positif        non
  Etat de jeu a 15 min               csdiffat15    numerique  Oracle's Elixir                            colonne brute          0.00         0.748 positif        non
  Etat de jeu a 15 min          diff_kills_at15    numerique          Derivee            killsat15 moins opp_killsat15          0.00         0.763 positif        non
  Etat d

### Ce que le tableau dit

**Cinq features portent presque tout le signal disponible à la minute 15.** L'écart d'or et sa
version normalisée atteignent 0,82 d'AUC à elles seules, suivies de l'écart d'expérience, de
l'écart de kills et de l'écart de sbires. C'est cohérent avec le métier : ces quatre écarts
mesurent la même domination économique par quatre canaux différents.

**Le plafond de 0,82 est en soi un contrôle de fuite.** Une seule variable qui monterait à 0,95
signifierait qu'elle contient le résultat. C'est exactement ce qui a fait écarter `firsttower`
et `damagetotowers` en phase 3.

**Les features de composition tournent autour de 0,50.** Prises seules, elles ne prédisent rien.
La phase 5 vérifiera si elles apportent quelque chose à avantage économique égal, ce qui est la
seule façon honnête de poser la question.

**Les features historiques sont modérées et c'est rassurant.** Une forme d'équipe à 0,60 d'AUC
est crédible pour une information extérieure à la partie. Si elle était montée à 0,80, il
faudrait suspecter que la fenêtre expansive lit le présent, malgré l'audit de la section 5.5.


## 9. Export vers `data/processed/`

In [17]:
config.DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
chemin = config.DATA_PROCESSED / "lol_at15.parquet"

final = final.sort_values(["date", "gameid", "side"]).reset_index(drop=True)
final.to_parquet(chemin, index=False)

print(f"{chemin.name} : {len(final):,} lignes x {final.shape[1]} colonnes "
      f"({chemin.stat().st_size / 1e6:.1f} Mo)")
print()

relu = pd.read_parquet(chemin)
print("Relecture :", relu.shape, "| identique :", relu.shape == final.shape)
print("Types préservés :", relu["date"].dtype, "|", relu["result"].dtype)
print()
print(f"Entraînement, avant {config.SPLIT_DATE} : "
      f"{int((relu['date'] < config.SPLIT_DATE).sum()):,} lignes")
print(f"Test, à partir du {config.SPLIT_DATE}   : "
      f"{int((relu['date'] >= config.SPLIT_DATE).sum()):,} lignes")

lol_at15.parquet : 92,616 lignes x 49 colonnes (5.8 Mo)

Relecture : (92616, 49) | identique : True
Types préservés : datetime64[ns] | int8

Entraînement, avant 2026-01-01 : 76,070 lignes
Test, à partir du 2026-01-01   : 16,546 lignes


## 10. Dictionnaire de données

Généré depuis le dataset, pour que la documentation ne puisse pas diverger du contenu réel.

In [18]:
DESCRIPTIONS = {
    "gameid": ("Identifiant de la partie", "Oracle's Elixir", "brute"),
    "teamid": ("Identifiant de l'équipe, Inconnu si absent", "Oracle's Elixir", "nettoyée"),
    "team_key": ("Clé d'équipe, teamid ou teamname en repli", "Dérivée", "construite"),
    "teamname": ("Nom de l'équipe", "Oracle's Elixir", "nettoyée"),
    "date": ("Date et heure de la partie", "Oracle's Elixir", "parsée format explicite"),
    "saison": ("Année civile de la date, porte le split", "Dérivée", "construite"),
    "league": ("Code de la ligue, analyse seulement", "Oracle's Elixir", "brute"),
    "patch": ("Version du jeu, texte", "Oracle's Elixir", "brute"),
    "result": ("Cible, 1 si l'équipe gagne", "Oracle's Elixir", "brute"),
    "golddiffat15": ("Écart d'or à 15 minutes", "Oracle's Elixir", "brute"),
    "xpdiffat15": ("Écart d'expérience à 15 minutes", "Oracle's Elixir", "brute"),
    "csdiffat15": ("Écart de sbires à 15 minutes", "Oracle's Elixir", "brute"),
    "diff_kills_at15": ("killsat15 moins opp_killsat15", "Dérivée", "calculée"),
    "deathsat15": ("Morts de l'équipe à 15 minutes", "Oracle's Elixir", "brute"),
    "objectifs_precoces": ("firstblood plus firstdragon plus firstherald, 0 à 3",
                           "Dérivée", "calculée"),
    "compo_nb_tank": ("Champions taggés Tank parmi les 5", "Data Dragon", "construite"),
    "compo_nb_mage": ("Champions taggés Mage parmi les 5", "Data Dragon", "construite"),
    "compo_nb_marksman": ("Champions taggés Marksman parmi les 5", "Data Dragon", "construite"),
    "compo_nb_fighter": ("Champions taggés Fighter parmi les 5", "Data Dragon", "construite"),
    "compo_nb_assassin": ("Champions taggés Assassin parmi les 5", "Data Dragon", "construite"),
    "compo_nb_support": ("Champions taggés Support parmi les 5", "Data Dragon", "construite"),
    "profil_degats": ("Part de champions AD parmi AD plus AP, 0 à 1",
                      "Data Dragon", "construite"),
    "forme_equipe_10_derniers": ("Taux de victoire sur les 10 parties précédentes",
                                 "Dérivée", "fenêtre expansive, passé seul"),
    "experience_roster": ("Parties déjà jouées par ce cinq", "Dérivée", "cumcount, passé seul"),
    "winrate_champion_patch": ("Taux de victoire moyen des 5 champions sur le patch, "
                               "parties antérieures", "Dérivée",
                               "fenêtre expansive, passé seul"),
    "ecart_or_normalise": ("golddiffat15 divisé par la médiane d'or du patch",
                           "Dérivée", "fenêtre expansive, passé seul"),
    "mediane_or_patch": ("Médiane d'or à 15 sur les parties antérieures du patch",
                         "Dérivée", "fenêtre expansive, passé seul"),
    "patch_seq": ("Rang du patch dans sa saison", "Dérivée", "construite"),
    "side": ("Côté de la carte, Blue ou Red", "Oracle's Elixir", "brute"),
    "region": ("Région du circuit", "Référentiel XLSX", "jointure"),
    "tier_ligue": ("Niveau de ligue, 1 à 3", "Référentiel XLSX", "jointure"),
    "playoffs": ("Phase finale ou saison régulière", "Oracle's Elixir", "brute"),
    "is_playoffs": ("Copie entière de playoffs", "Dérivée", "calculée"),
    "firstblood": ("Premier sang obtenu", "Oracle's Elixir", "brute"),
    "firstdragon": ("Premier dragon obtenu", "Oracle's Elixir", "brute"),
    "firstherald": ("Premier héraut obtenu", "Oracle's Elixir", "brute"),
    "mois": ("Mois de la partie, analyse seulement", "Dérivée", "construite"),
    "year": ("Étiquette de saison d'Oracle's Elixir, analyse seulement",
             "Oracle's Elixir", "brute"),
    "patch_major": ("Partie majeure du patch, analyse seulement", "Dérivée", "construite"),
    "patch_minor": ("Partie mineure du patch, analyse seulement", "Dérivée", "construite"),
    "turretplates": ("Plaques prises, hors features car l'échelle change en 2026",
                     "Oracle's Elixir", "brute"),
    "flag_ecart_or_extreme": ("Écart d'or hors bornes IQR", "Dérivée", "drapeau"),
    "confiance": ("Fiabilité du classement de la ligue", "Référentiel XLSX", "jointure"),
    "franchisee": ("Ligue franchisée", "Référentiel XLSX", "jointure"),
    "roster_key": ("Identifiant du cinq de départ", "Dérivée", "construite"),
    "goldat15": ("Or de l'équipe à 15 minutes", "Oracle's Elixir", "brute"),
    "opp_goldat15": ("Or adverse à 15 minutes", "Oracle's Elixir", "brute"),
    "xpat15": ("Expérience de l'équipe à 15 minutes", "Oracle's Elixir", "brute"),
    "csat15": ("Sbires de l'équipe à 15 minutes", "Oracle's Elixir", "brute"),
}

features_modele_set = set(config.NUMERIC_FEATURES + config.CATEGORICAL_FEATURES
                          + config.BOOLEAN_FEATURES)

lignes = []
for col in final.columns:
    desc, source, transfo = DESCRIPTIONS.get(col, ("A documenter", "?", "?"))
    lignes.append({
        "Colonne": f"`{col}`",
        "Type": str(final[col].dtype),
        "Description": desc,
        "Source": source,
        "Transformation": transfo,
        "Feature modèle": "oui" if col in features_modele_set else "non",
        "Manquant (%)": round(100 * final[col].isna().mean(), 2),
    })
dictionnaire = pd.DataFrame(lignes)


def tableau_markdown(df):
    entete = "| " + " | ".join(str(c) for c in df.columns) + " |"
    sep = "|" + "|".join(["---"] * len(df.columns)) + "|"
    corps = ["| " + " | ".join(str(v) for v in row) + " |"
             for row in df.itertuples(index=False)]
    return "\n".join([entete, sep] + corps)


doc = f"""# Data dictionary

Dataset final : `data/processed/lol_at15.parquet`

Document généré par `notebooks/04_transformation.ipynb`, à ne pas éditer à la main.

Une ligne est une équipe dans une partie. Deux lignes par partie, identifiées par `gameid`
plus `side`.

| Propriété | Valeur |
|---|---|
| Lignes | {len(final):,} |
| Colonnes | {final.shape[1]} |
| Parties | {final['gameid'].nunique():,} |
| Période | {final['date'].min():%Y-%m-%d} au {final['date'].max():%Y-%m-%d} |
| Cible | `result`, équilibrée à {100 * final['result'].mean():.1f} % |
| Entraînement | {int((final['date'] < config.SPLIT_DATE).sum()):,} lignes, avant le {config.SPLIT_DATE} |
| Test | {int((final['date'] >= config.SPLIT_DATE).sum()):,} lignes |

## Instant de prédiction

Toutes les colonnes de ce dataset sont connues à la **minute 15**. Les colonnes postérieures
ont été supprimées en phase 3, liste `config.LEAKY_COLUMNS`, et le rapport de nettoyage détaille
les cinq oublis rattrapés par un contrôle de corrélation.

## Features historiques et fuite temporelle

Cinq colonnes regardent le passé : `forme_equipe_10_derniers`, `experience_roster`,
`winrate_champion_patch`, `mediane_or_patch` et `ecart_or_normalise` qui en dérive.

Toutes sont calculées en fenêtre expansive sur les parties **antérieures** uniquement, triées
par date, au moyen d'un `shift(1)` ou d'un `cumcount`. Le code est dans `src/features.py`, et
l'audit de la section 5.5 du notebook vérifie qu'aucune n'est renseignée sur la première partie
d'un groupe.

Leurs valeurs manquantes sont des amorçages et ne sont **pas** imputées ici. L'imputation aura
lieu dans le `Pipeline` de la phase 7, ajusté sur le seul jeu d'entraînement.

## Colonnes

{tableau_markdown(dictionnaire)}

## Colonnes volontairement absentes des features

| Colonne | Raison |
|---|---|
| `league`, `year`, `patch_major` | Modalités qui ne survivent pas à la frontière 2025/2026 |
| `split` | 32 modalités, 19,8 % de manquants, 3 absentes du train |
| `turretplates` | Échelle qui change en 2026, maximum de 15 à 45 |
| `firsttower` | Drapeau de fin de partie, écarté pour fuite en phase 3 |
| `teamname`, `teamid`, `roster_key` | Identifiants, servent aux features de forme uniquement |
| `mois` | Aucun sens prédictif, conservée pour l'analyse de phase 5 |
"""

chemin_doc = config.DOCS / "data_dictionary.md"
chemin_doc.write_text(doc, encoding="utf-8")
print(f"Dictionnaire écrit : {chemin_doc}")
print(f"{len(doc):,} caractères, {len(dictionnaire)} colonnes documentées")
print(f"Colonnes non documentées : "
      f"{int((dictionnaire['Description'] == 'A documenter').sum())}")

Dictionnaire écrit : E:\LWP(LoLWinPrediciton)\lol-win-prediction\docs\data_dictionary.md
7,204 caractères, 49 colonnes documentées
Colonnes non documentées : 0


## 11. Réflexion

**La feature la plus prometteuse.** `ecart_or_normalise`, parce qu'elle répond directement à la
dérive mesurée en phase 2. L'écart d'or brut est le meilleur signal disponible, mais sa valeur
se déplace d'une saison à l'autre. La version normalisée porte la même information sous une
forme qui traverse la frontière du split.

**Les jointures.** Aucune perte : ce sont des `left` à cardinalité vérifiée par `validate`, et
le compte de lignes est identique avant et après. Le choix de `gameid` plus `side` comme clé,
plutôt que `gameid` plus `teamid`, a évité de fusionner des équipes non identifiées.

**L'équilibre du dataset.** Seize features numériques, trois catégorielles et quatre booléennes
pour 92 616 lignes : le rapport est confortable. Une redondance connue subsiste, `golddiffat15`
étant exactement `goldat15` moins `opp_goldat15`, et `ecart_or_normalise` en dérivant. La
phase 7 tranchera avec les coefficients de la régression logistique.

**Ce qui manque.** L'ordre de la draft, absent d'Oracle's Elixir, qui porterait une vraie
information stratégique. Et un historique des tags de champions : ceux de Data Dragon décrivent
le patch courant, alors qu'un champion joué en support en 2022 mais classé Marksman aujourd'hui
sera mal décrit rétrospectivement.